<a href="https://colab.research.google.com/github/tangitapkullaniyor/CENG467_Midterm_290201060/blob/main/Question4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers evaluate sacrebleu bert-score nltk sentencepiece -q

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from datasets import load_dataset

dataset_mt = load_dataset("opus_books", "de-en")

print(dataset_mt)
print(dataset_mt["train"][0])

In [ ]:
data = dataset_mt["train"].shuffle(seed=42).select(range(3000))

split = data.train_test_split(test_size=0.2, seed=42)
train_data = split["train"]
temp_data = split["test"]

split2 = temp_data.train_test_split(test_size=0.5, seed=42)
val_data = split2["train"]
test_data = split2["test"]

print(len(train_data), len(val_data), len(test_data))

In [ ]:
def get_src_tgt(example):
    return example["translation"]["en"], example["translation"]["de"]

src_example, tgt_example = get_src_tgt(train_data[0])

print("EN:", src_example)
print("DE:", tgt_example)

In [ ]:
import re

def simple_tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"([.!?,])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

In [ ]:
from collections import Counter

PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

special_tokens = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

def build_vocab(sentences, max_vocab_size=10000, min_freq=2):
    counter = Counter()

    for sent in sentences:
        counter.update(simple_tokenize(sent))

    vocab = special_tokens.copy()

    for word, freq in counter.most_common(max_vocab_size):
        if freq >= min_freq and word not in vocab:
            vocab.append(word)

    stoi = {word: i for i, word in enumerate(vocab)}
    itos = {i: word for word, i in stoi.items()}

    return stoi, itos

train_src_texts = [get_src_tgt(x)[0] for x in train_data]
train_tgt_texts = [get_src_tgt(x)[1] for x in train_data]

src_stoi, src_itos = build_vocab(train_src_texts)
tgt_stoi, tgt_itos = build_vocab(train_tgt_texts)

print("Source vocab size:", len(src_stoi))
print("Target vocab size:", len(tgt_stoi))

In [ ]:
PAD_IDX = src_stoi[PAD_TOKEN]
SOS_IDX = tgt_stoi[SOS_TOKEN]
EOS_IDX = tgt_stoi[EOS_TOKEN]
UNK_SRC_IDX = src_stoi[UNK_TOKEN]
UNK_TGT_IDX = tgt_stoi[UNK_TOKEN]

MAX_LEN = 30

def encode_src(text):
    tokens = simple_tokenize(text)[:MAX_LEN]
    ids = [src_stoi.get(tok, UNK_SRC_IDX) for tok in tokens]
    return ids

def encode_tgt(text):
    tokens = simple_tokenize(text)[:MAX_LEN]
    ids = [SOS_IDX] + [tgt_stoi.get(tok, UNK_TGT_IDX) for tok in tokens] + [EOS_IDX]
    return ids

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src, tgt = get_src_tgt(self.data[idx])
        return torch.tensor(encode_src(src), dtype=torch.long), torch.tensor(encode_tgt(tgt), dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    src_batch = pad_sequence(src_batch, padding_value=src_stoi[PAD_TOKEN], batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=tgt_stoi[PAD_TOKEN], batch_first=True)

    return src_batch, tgt_batch

train_loader = DataLoader(TranslationDataset(train_data), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(TranslationDataset(val_data), batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(TranslationDataset(test_data), batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=src_stoi[PAD_TOKEN])
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden


class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return F.softmax(attention, dim=1)


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=tgt_stoi[PAD_TOKEN])
        self.attention = Attention(hid_dim)
        self.rnn = nn.GRU(emb_dim + hid_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear(emb_dim + hid_dim + hid_dim, output_dim)

    def forward(self, input_token, hidden, encoder_outputs):
        input_token = input_token.unsqueeze(1)

        embedded = self.embedding(input_token)

        attn_weights = self.attention(hidden, encoder_outputs)
        attn_weights = attn_weights.unsqueeze(1)

        context = torch.bmm(attn_weights, encoder_outputs)

        rnn_input = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(rnn_input, hidden)

        prediction = self.fc_out(torch.cat((output.squeeze(1), context.squeeze(1), embedded.squeeze(1)), dim=1))

        return prediction, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        encoder_outputs, hidden = self.encoder(src)

        input_token = tgt[:, 0]

        for t in range(1, tgt_len):
            output, hidden = self.decoder(input_token, hidden, encoder_outputs)
            outputs[:, t] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input_token = tgt[:, t] if teacher_force else top1

        return outputs

In [ ]:
INPUT_DIM = len(src_stoi)
OUTPUT_DIM = len(tgt_stoi)
EMB_DIM = 128
HID_DIM = 256

encoder = Encoder(INPUT_DIM, EMB_DIM, HID_DIM)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM)

seq2seq_model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = torch.optim.Adam(seq2seq_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi[PAD_TOKEN])

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    epoch_loss = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()

        output = model(src, tgt)

        output_dim = output.shape[-1]

        output = output[:, 1:].reshape(-1, output_dim)
        tgt = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)


def evaluate_loss(model, loader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)

            output = model(src, tgt, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]

            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt)

            epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [ ]:
N_EPOCHS = 5

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(seq2seq_model, train_loader, optimizer, criterion)
    val_loss = evaluate_loss(seq2seq_model, val_loader, criterion)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

In [ ]:
def translate_seq2seq(sentence, max_len=30):
    seq2seq_model.eval()

    src_ids = torch.tensor([encode_src(sentence)], dtype=torch.long).to(device)

    with torch.no_grad():
        encoder_outputs, hidden = seq2seq_model.encoder(src_ids)

    input_token = torch.tensor([tgt_stoi[SOS_TOKEN]], dtype=torch.long).to(device)

    outputs = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden = seq2seq_model.decoder(input_token, hidden, encoder_outputs)

        pred_token = output.argmax(1).item()

        if pred_token == tgt_stoi[EOS_TOKEN]:
            break

        outputs.append(tgt_itos.get(pred_token, UNK_TOKEN))
        input_token = torch.tensor([pred_token], dtype=torch.long).to(device)

    return " ".join(outputs)

In [ ]:
src, ref = get_src_tgt(test_data[0])
pred_seq2seq = translate_seq2seq(src)

print("SOURCE:", src)
print("REFERENCE:", ref)
print("SEQ2SEQ:", pred_seq2seq)

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

transformer_name = "Helsinki-NLP/opus-mt-en-de"

marian_tokenizer = MarianTokenizer.from_pretrained(transformer_name)
marian_model = MarianMTModel.from_pretrained(transformer_name).to(device)

In [ ]:
def translate_transformer(sentence):
    inputs = marian_tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        translated = marian_model.generate(**inputs, max_length=128)

    return marian_tokenizer.decode(translated[0], skip_special_tokens=True)

In [ ]:
pred_transformer = translate_transformer(src)

print("SOURCE:", src)
print("REFERENCE:", ref)
print("SEQ2SEQ:", pred_seq2seq)
print("TRANSFORMER:", pred_transformer)

In [ ]:
eval_subset = test_data.select(range(100))

sources = []
references = []
seq2seq_preds = []
transformer_preds = []

for example in eval_subset:
    src, ref = get_src_tgt(example)

    sources.append(src)
    references.append(ref)

    seq2seq_preds.append(translate_seq2seq(src))
    transformer_preds.append(translate_transformer(src))

print("Done")

In [ ]:
import evaluate

bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
chrf = evaluate.load("chrf")
bertscore = evaluate.load("bertscore")

In [ ]:
bleu_seq2seq = bleu.compute(predictions=seq2seq_preds, references=[[r] for r in references])
bleu_transformer = bleu.compute(predictions=transformer_preds, references=[[r] for r in references])

print("Seq2Seq BLEU:", bleu_seq2seq)
print("Transformer BLEU:", bleu_transformer)

In [ ]:
meteor_seq2seq = meteor.compute(predictions=seq2seq_preds, references=references)
meteor_transformer = meteor.compute(predictions=transformer_preds, references=references)

print("Seq2Seq METEOR:", meteor_seq2seq)
print("Transformer METEOR:", meteor_transformer)

In [ ]:
chrf_seq2seq = chrf.compute(predictions=seq2seq_preds, references=references)
chrf_transformer = chrf.compute(predictions=transformer_preds, references=references)

print("Seq2Seq ChrF:", chrf_seq2seq)
print("Transformer ChrF:", chrf_transformer)

In [ ]:
bertscore_seq2seq = bertscore.compute(
    predictions=seq2seq_preds,
    references=references,
    lang="de"
)

bertscore_transformer = bertscore.compute(
    predictions=transformer_preds,
    references=references,
    lang="de"
)

print("Seq2Seq BERTScore F1:", np.mean(bertscore_seq2seq["f1"]))
print("Transformer BERTScore F1:", np.mean(bertscore_transformer["f1"]))

In [ ]:
for i in range(3):
    print("=" * 80)
    print("EXAMPLE", i+1)
    print("SOURCE:", sources[i])
    print("REFERENCE:", references[i])
    print("SEQ2SEQ:", seq2seq_preds[i])
    print("TRANSFORMER:", transformer_preds[i])